<a href="https://colab.research.google.com/github/Hiarley007/Marchine_Learning/blob/main/lab3_HIARLEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Conectando google colab com o google drive

try:
    # O módulo google.colab SÓ existe dentro do Colab. É por isso que a
    # importação fica dentro de um try: se você um dia abrir este mesmo
    # notebook no seu computador, o código não quebra.
    from google.colab import drive

    # mount() "pendura" o seu Drive dentro da pasta /content/drive do
    # runtime. Depois disso, o Drive vira uma pasta comum para o Python.
    drive.mount('/content/drive')

    NO_COLAB = True
    print("Drive montado com sucesso.")

except ModuleNotFoundError:
    # Caímos aqui se não estamos no Colab. Não é erro, é só um aviso.
    NO_COLAB = False
    print("Não estamos no Colab: vou procurar o arquivo na pasta local.")

Mounted at /content/drive
Drive montado com sucesso.


In [ ]:
# ---------------------------------------------------------------------
# Objetivo desta célula: montar o caminho completo do arquivo e conferir
# se ele existe, ANTES de tentar ler.
#
# Variáveis criadas:
#   PASTA   (str) -> a pasta onde o arquivo está
#   ARQUIVO (str) -> o nome do arquivo
#   CAMINHO (str) -> os dois juntos, que é o que o pandas vai receber
# ---------------------------------------------------------------------

import os   # módulo de utilidades do sistema de arquivos

# Se você criou a pasta com outro nome, é ESTA linha que você altera.
PASTA = '/content/drive/MyDrive/Aprendizagem_de_maquinas/' if NO_COLAB else ''
ARQUIVO = 'mcdonalds_x_burger_king.csv'

# os.path.join junta pasta e arquivo cuidando das barras. É mais seguro
# do que somar textos com "+", que erra quando falta ou sobra uma barra.
CAMINHO = os.path.join(PASTA, ARQUIVO)

print("Caminho montado:", CAMINHO)

# exists() devolve True ou False. Nunca lança erro — só responde.
if os.path.exists(CAMINHO):
    print("Arquivo encontrado. Pode seguir.")
else:
    print("Arquivo NÃO encontrado. Confira o nome da pasta e do arquivo.")

Caminho montado: /content/drive/MyDrive/Aprendizagem_de_maquinas/mcdonalds_x_burger_king.csv
Arquivo encontrado. Pode seguir.


## Etapa 01 - A leitura ingênua — e por que ela falha três vezes

Agora vem a parte que quase ninguém ensina, porque parece trivial: **ler o arquivo**.

A função é `pd.read_csv()`. A sigla CSV significa *Comma-Separated Values* — valores separados por vírgula. O problema é que essa promessa quase nunca se cumpre. Arquivos gerados no Brasil, na Europa ou por planilhas configuradas em português usam **ponto e vírgula** como separador, porque a vírgula já está ocupada: ela é o separador decimal.

Vamos ler o arquivo três vezes, errando de propósito, e consertar um problema por vez.

In [ ]:
import pandas as pd

try:
  tentativa_01 = pd.read_csv(CAMINHO)
  print("Leu sem erro. Formato:", tentativa_01.shape)
except pd.errors.ParserError as erro:
  print("Erro de leitura. A mensagem foi:\n")
  print(erro)

Erro de leitura. A mensagem foi:

Error tokenizing data. C error: Expected 2 fields in line 17, saw 3



# Diagnóstico 01 - Erro na linha 17

Logo de cara aponta o sintoma e não a causa. Mais é bem claro, ele reclama que, na linha 17, pois quando aprece assim, o problema é na linha toda.

Com isso, precisamos reconstruir linha por linha. Que é da seguinte forma, há vários valores com vírgulas. E com isso, o pandas não sabe se é um número quebrado.

E, para resolver isso, utilizamos o separador (sep = ";") para consertar o parâmetro.

In [ ]:
# TENTATIVA 02 - CORRIGINDO O SEPARADOR

tentativa_02 = pd.read_csv(CAMINHO, sep=";")
print("Formato lido:", tentativa_02.shape)

print("\nTipo de cada coluna:")
print(tentativa_02.dtypes)

Formato lido: (499, 15)

Tipo de cada coluna:
Chain                object
Item                 object
Type                 object
Serving Size (g)     object
Calories              int64
Calories from fat     int64
Total Fat (g)        object
Saturated Fat (g)    object
Trans Fat (g)        object
Chol (mg)             int64
Sodium (mg)           int64
Total Carb (g)        int64
Dietary Fiber (g)     int64
Total Sugar (g)       int64
Protein (g)           int64
dtype: object


# Diagnóstico 02 - Quatro colunas de valores númericos estão como texto.


As respectivas colunas, são:
- Serving Size (g);
- Total Fat (g);
- Saturated Fat (g);
- Trans Fat (g).

Essas colunas não reconhecidas com valor númerico por ter presente valores quebrados, escritos em decimal com vírgula

In [ ]:
# TENTATIVA 03 - Avisando ao Pandas que a vírgula é decimal

# decimal = '-' para interpretar valores como: 1,6...

tentativa_03 = pd.read_csv(CAMINHO, sep=';', decimal=',')
print("Tipos corrigidos após o parâmetro decimal:\n")
print(tentativa_03.dtypes)

Tipos corrigidos após o parâmetro decimal:

Chain                 object
Item                  object
Type                  object
Serving Size (g)     float64
Calories               int64
Calories from fat      int64
Total Fat (g)        float64
Saturated Fat (g)     object
Trans Fat (g)         object
Chol (mg)              int64
Sodium (mg)            int64
Total Carb (g)         int64
Dietary Fiber (g)      int64
Total Sugar (g)        int64
Protein (g)            int64
dtype: object


# Diagnóstico 03 - Duas colunas ainda persitem não mudar para numéricos.


**As duas colunas que viraram float foi:**
 - Serving Size (g);
 - Total Fat (g).

**E as duas que resistiram foi:**
 - Saturated Fat (g);
 - Trans Fat (g)

Com isso, precisamos fazer um diagnóstico para descobrir porque ed=ssas colunas ainda persistem em não virar numericas.

In [ ]:
# ---------------------------------------------------------------------
# Objetivo desta célula: descobrir QUAL valor está impedindo a conversão.
#
# Estratégia: pd.to_numeric com errors='coerce' converte o que dá e
# transforma o que não dá em NaN. Filtrando pelos NaN, sobram exatamente
# os valores problemáticos.
#
# É a técnica geral para "por que esta coluna não vira número?".
# ---------------------------------------------------------------------

coluna = tentativa_03['Trans Fat (g)']

# to_numeric tenta converter; 'coerce' manda virar NaN o que falhar.
# Trocamos a vírgula por ponto ANTES de testar. Sem isso, os valores
# como '1,5' também falhariam na conversão e entrariam na lista,
# escondendo o problema NOVO no meio do problema que já conhecemos.
convertida = pd.to_numeric(coluna.str.replace(',', '.'), errors='coerce')

# .unique() lista os valores distintos que não passaram na conversão.
problematicos = coluna[convertida.isna()].unique()

print("Valores que impedem a conversão:")
for v in problematicos[:8]:
    # repr() de novo: precisamos ver os espaços.
    print("   ", repr(v))


Valores que impedem a conversão:
    ' -   '


# O que esta impedindo é: (' - '), o hífen, traço contábil do Excel.



Isso não é um lixo aleatório: pois é o formato contábil do Excel, onde exibe valor zero alinhado a direita. E, além desse, pode ser outros, como alguns caracteres especiais e até mesmo combinações de letras, números e caracteres especiais.


In [ ]:
# Objetivo desta célula: descobrir o que o traço significa,
# em cada uma das colunas.

for nome in ['Saturated Fat (g)', 'Trans Fat (g)']:
  tem_traco = tentativa_03[nome].str.strip() == '-'

print(tem_traco)

gordura = tentativa_03.loc[tem_traco, 'Total Fat (g)']

print(f"{nome}")
print(f"Itens com traço: {tem_traco.sum()}")
print(f"Total de gordura dos itens ->"
      f"média {gordura.mean():.2f} g, máximo {gordura.max():.1f} g\n")

0      False
1      False
2      False
3      False
4      False
       ...  
494    False
495    False
496    False
497    False
498    False
Name: Trans Fat (g), Length: 499, dtype: bool
Trans Fat (g)
Itens com traço: 36
Total de gordura dos itens ->média 21.94 g, máximo 74.0 g



# O memso símbolo, mas com significados opostos

Quando é feita essa comparação, o gráfico possui dois significados:

| Coluna | Itens com traço | Gordura total desses itens |
|---|---|---|
| `Saturated Fat (g)` | 83 | média 0,16 g — **máximo 3,0 g** |
| `Trans Fat (g)` | 36 | média 21,9 g — **máximo 74,0 g** |

→ **Saturated Fat:** são elementos que quase não têm gordura, onde o zero nesse leite é considerado **defensável**.

→ **Trans Fat:** o máximo é de **74,0 g** e a média é de **21,9 g**, muito acima da média. Então, vê-se que o zero é mais favorável ou pode ser deixado em branco.

**Passo prático:** quando a tabela estiver com (-), faça uma nova análise. Compare os valores e, se o resultado for próximo de **0**, é **defensável**. Caso seja distante, faça uma nova análise novamente.


# Coodificação: forma correta de tranforma as letras em bytes

Leitura ERRADA nos dias atuais:

**econding = latin-1**



Leitura CORRETA, descartando explicamnete o que é chamado de BOM.

**ecoding = utf-8-sig'**

In [ ]:
# Testando os dois:

errada = pd.read_csv(CAMINHO, sep=';', decimal=',', encoding='latin-1')
print("Com o latin-1", repr(errada.columns[0]))

correto = pd.read_csv(CAMINHO, sep=';', decimal=',',encoding='utf-8-sig')
print("Com UTF-8", repr(correto.columns[0]))

Com o latin-1 'ï»¿Chain'
Com UTF-8 'Chain'


# Leitura definitiva com os 4 parâmetros de forma definitiva.

In [ ]:
# Leitura de forma único para os 4 parâmetros para um arquivo limpo.

# sep=';'               -> separador de colunas;
# decimal='-'           -> separador decimal;
# na_values=[' -    ']  -> traço contábil, é contado como ausente;
# encoding='utf-8-sig'  -> descarta a marca BOM no início do arquivo.


df = pd.read_csv (
    CAMINHO,
    sep=';',
    decimal=',',
    na_values=[' -   ', '-', ' -'],
    encoding='utf-8-sig',
)

print("Formato:", df.shape)
print("Primeira Coluna:", repr(df.columns[0]))
print("\n Tipos das 4 Colunas problemáticas:")
print(df[['Serving Size (g)', 'Total Fat (g)',
          'Saturated Fat (g)', 'Trans Fat (g)']].dtypes)

Formato: (499, 15)
Primeira Coluna: 'Chain'

 Tipos das 4 Colunas problemáticas:
Serving Size (g)     float64
Total Fat (g)        float64
Saturated Fat (g)    float64
Trans Fat (g)        float64
dtype: object


# Etapa 02 - NumPy

NumPy é a base do Pandas — toda coluna numérica de um DataFrame é, por baixo, um vetor NumPy.

A diferença central: em Python puro, um for aplica a operação elemento por elemento, no interpretador, com toda a lentidão que isso implica. Em NumPy, a operação (vetor * 1.1) é enviada de uma vez para rotinas compiladas em C, que processam o vetor inteiro em bloco.

Isso é vetorização — mais rápido, mais curto, e a razão pela qual operações pesadas (como treinar redes neurais) são viáveis em Python.

In [ ]:
# Objetivo dessa aula é: medir a diferença do processamento de informação
# entre o Python Puro com o (for) e a vetorização da Biblioteca Numpy.

# Com isso, teremos duas variavéis, onde:
# 1 - Será para o Python puro, chamada de : valores_lista;
# 2 - e a outra, será para o NumPy com vetor, chamdade de: valores_array.

import numpy as np
import time

N = 1_000_000
# .list() - lista os numeros e materializa na memoria
# .range() - gera os numeros.
valores_lista = list(range(N))

# .np.array() - converte essa lista em vertor NumPy
valores_array = np.array(valores_lista)

# -------------- Medição 01: Utilizando for no Python Puro --------------------
inicio = time.perf_counter()
resultado_lista = [x * 1.1 for x in valores_lista]
tempo_python = time.perf_counter() - inicio

# -------------- Medição 02: Utilizando NumPy com vetor --------------------
inicio = time.perf_counter()
resultado_array = valores_array * 1.1
tempo_numpy = time.perf_counter() - inicio

# Tempo de cada um e qual o mais rápido.
print(f"for cdo Python Puro: {tempo_python:.4f} segundos")
print(f"NumPy com vetor    : {tempo_numpy:.4f} segundos")
print(f"NumPy foi          : {tempo_python / tempo_numpy:.0f} vezes mais rápido")

for cdo Python Puro: 0.0802 segundos
NumPy com vetor    : 0.0051 segundos
NumPy foi          : 16 vezes mais rápido


# Termo Importante!!!


**Broadcasting** - é quando você opera diversas posições com um único número. E com isso, ao invés, de usar loop, ele espalha o próprio vetor e que já faz a leitura de cada elemento de forma automática.

In [ ]:
# Objetivo desta célula - demostrar braodcsting e as operações que
# usaremos no resto da aula.

# Um vetor pequeno para de fácil entendimento.
calorias = np.array([250, 540, 1220, 190, 370])
print("Vetor Original: ", calorias)

# Broadcasting: o escalar 2 sendo espalhado pelas 5 posições
print("Dividido por 2     :", calorias / 2)

# Comparação de valores, também é vetorizada, onde devolve um boolean
mascara = calorias > 500
print("Maiores que 500?    :", mascara)

# Indexação booleana: usar o vetor de True/False para SELECIONAR.
print("Só os maiores que 500:", calorias[mascara])

# Funções de agrgação, que reduzem o vetor a um número só
print(f"\nMédia: {calorias.mean():.1f} | Mediana: {np.median(calorias):.1f}")
print(f"\nDesvião Padrão: {calorias.std(ddof=1):.1f}")

Vetor Original:  [ 250  540 1220  190  370]
Dividido por 2     : [125. 270. 610.  95. 185.]
Maiores que 500?    : [False  True  True False False]
Só os maiores que 500: [ 540 1220]

Média: 514.0 | Mediana: 370.0

Desvião Padrão: 416.7


# Etapa 03 - Nomeclatura do DataFrame

o DataFrame é uma tabela, que possui:

- Linhas;
- Colunas Nomeadas;
- Um indíce;

E antes de qualquer atividade de produção com os dados, há 5 comandos quue são essenciais, pois é que faz o reconhecimento de uma base nova de dados.

**VEREMOS NO CÓDIGO DE UM POR UM**

In [ ]:
# Ritual de conhecimento de uma base de dados nova
# Os comando são:

# 1. .shape -> tamanho da base de dados em linha e colunas;
# 2. head() -> como os dados são de verdade e devolde quantidade de linhas (n);
# 3. columns -> quais colunas existem;
# 4. info() -> tipo e quantidades de preenchidos.
# .describe() -> o resumo estatístico.


# 1 COMANDO -> .shape
print("1) TAMANHO")
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
print()
# 2 COMANDO - .head()
print("2) AS 5 PRIMEIRAS LINHAS 'OBRIGATORIAMENTE'")
display(df.head())
print()

# COMAMNDO .info() - onde traz informações detalhadas, ou seja, comando mais informativo
df.info()

1) TAMANHO
Linhas: 499 | Colunas: 15

2) AS 5 PRIMEIRAS LINHAS 'OBRIGATORIAMENTE'


,Chain,Item,Type,Serving Size (g),Calories,Calories from fat,Total Fat (g),Saturated Fat (g),Trans Fat (g),Chol (mg),Sodium (mg),Total Carb (g),Dietary Fiber (g),Total Sugar (g),Protein (g)
0,Burger King,Whopper Sandwich,Whopper Sandwiches,270.0,660,360,40.0,12.0,1.5,90,980,49,2,11,28
1,Burger King,Whopper Sandwich with Cheese,Whopper Sandwiches,292.0,740,420,46.0,16.0,2.0,115,1340,50,2,11,32
2,Burger King,Bacon & Cheese Whopper Sandwich,Whopper Sandwiches,303.0,790,460,51.0,17.0,2.0,125,1560,50,2,11,35
3,Burger King,Double Whopper Sandwich,Whopper Sandwiches,354.0,900,520,58.0,20.0,3.0,175,1050,49,2,11,48
4,Burger King,Double Whopper Sandwich with Cheese,Whopper Sandwiches,377.0,980,580,64.0,24.0,3.0,195,1410,50,2,11,52



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Chain              499 non-null    object 
 1   Item               499 non-null    object 
 2   Type               499 non-null    object 
 3   Serving Size (g)   241 non-null    float64
 4   Calories           499 non-null    int64  
 5   Calories from fat  499 non-null    int64  
 6   Total Fat (g)      499 non-null    float64
 7   Saturated Fat (g)  416 non-null    float64
 8   Trans Fat (g)      463 non-null    float64
 9   Chol (mg)          499 non-null    int64  
 10  Sodium (mg)        499 non-null    int64  
 11  Total Carb (g)     499 non-null    int64  
 12  Dietary Fiber (g)  499 non-null    int64  
 13  Total Sugar (g)    499 non-null    int64  
 14  Protein (g)        499 non-null    int64  
dtypes: float64(4), int64(8), object(3)
memory usage: 58.6+ KB


A maioria das colunas apresentam valores igual a 499 e tem três que dizem outras coisas, que é o seguinte:

> `Serving Size (g)    241 non-null`

> `Saturated Fat (g)   416 non-null`

> `Trans Fat (g)       463 non-null`

Há uma distância bem grande entre o 241 e 499, que se dá pelo fato dos traços no formato contábil no excel. E menos da metade tem esse peso na porção informada.Mas essa análise é mais para frente, na etapa 5.


>  `.describe()` -> resumo estatístico, onde faz o resuma das principais medidas estatísticas.

Onde calcula: **count, mean, std, min, 25%, 50%, 75%, min**


In [ ]:
# Apresentação na mão:

display(df.describe().T.round(1))

,count,mean,std,min,25%,50%,75%,max
Serving Size (g),241.0,183.8,131.4,0.7,97.0,154.0,249.0,713.0
Calories,499.0,313.0,242.7,0.0,150.0,260.0,420.0,1220.0
Calories from fat,499.0,111.3,134.0,0.0,0.0,70.0,170.0,750.0
Total Fat (g),499.0,12.4,14.9,0.0,0.0,8.0,18.0,84.0
Saturated Fat (g),416.0,5.8,5.8,0.0,1.5,4.0,8.0,33.0
Trans Fat (g),463.0,0.2,0.6,0.0,0.0,0.0,0.0,4.5
Chol (mg),499.0,42.0,79.6,0.0,0.0,15.0,40.0,575.0
Sodium (mg),499.0,411.7,508.7,0.0,85.0,160.0,600.0,2840.0
Total Carb (g),499.0,41.6,32.7,0.0,20.0,35.0,53.0,203.0
Dietary Fiber (g),499.0,1.0,1.5,0.0,0.0,0.0,2.0,9.0


# Etapa 04 - Selecionar, fatiar e filtrar em uma tabela


Em um vocabulário básico dos Pandas, precisa estar as 4 formas de pegar partes de uma tabela, que são as seguintes:

- **df['coluna´]** - seleciona uma coluna por vez que também é chamada de Series.

- **df[['a', 'b']]** - seleciona duas ou mais colunas e devolve um DataFrame;

- **df.loc[linhas, colunas]** - seleciona por rótulos (nome);

- **df.iloc[linhas, colunas]** - seleciona por posição (número).

Com esse vocabulários já se torna capas de brincar um pouco com as tabelas para extrair dados de maeira específica.

In [ ]:
# Comparando as 4 formas de seleção de dados de uma coluna no mesmo DataFrame.

# UMA COLUNA: devolve uma Series - vetor com indice
uma_coluna = df['Calories']
print("df['Calories'] devolve:", type(uma_coluna).__name__)
print(uma_coluna.head(3))

# Várias Colunas
varias_colunas = df[['Item', 'Calories']]
print("df[['Item' e 'Calories']]:", type(varias_colunas).__name__)

# .loc -> buscar colunas por rótulos(nome):
print("loc[0:2, 'Item'] ->", len(df.loc[0:2 , 'Item']), "linhas")

# .iloc -> buscar por posição (números):
print("iloc[0:2, 1]    ->", len(df.iloc[0:2, 1]), "linhas")

df['Calories'] devolve: Series
0    660
1    740
2    790
Name: Calories, dtype: int64
df[['Item' e 'Calories']]: DataFrame
loc[0:2, 'Item'] -> 3 linhas
iloc[0:2, 1]    -> 2 linhas


In [ ]:
# Passo 01: O resultado é a máscara.
mascara_pesados = df['Calories'] > 800

print("A máscara é uma Series de booleanos:")
print(mascara_pesados.head(5))

# Fazer uma soma, onde irá contar quantos booleanos acima de 800 tem na coluna
# Calories
print("Itens acima de 800 kcal em 'Calories':", mascara_pesados.sum())

# Utilizar mascara para filtrar
pesados = df[mascara_pesados]

display(pesados.nlargest(14, 'Calories')[['Chain', 'Item', 'Calories', 'Sodium (mg)']])

A máscara é uma Series de booleanos:
0    False
1    False
2    False
3     True
4     True
Name: Calories, dtype: bool
Itens acima de 800 kcal em 'Calories': 20


,Chain,Item,Calories,Sodium (mg)
6,Burger King,Triple Whopper Sandwich with Cheese,1220,1470
9,Burger King,Cheddar Bacon King Sandwich,1190,1930
298,Mc Donalds,Chocolate Triple Thick Shake (XLarge),1160,510
8,Burger King,Bacon King Sandwich,1150,2150
262,Mc Donalds,Big Breakfast with Hotcakes (Large),1150,2260
5,Burger King,Triple Whopper Sandwich,1130,1120
307,Mc Donalds,Strawberry Triple Thick Shake (XLarge),1110,350
311,Mc Donalds,Vanilla Triple Thick Shake (XLarge),1110,370
261,Mc Donalds,Big Breakfast with Hotcakes (Regular),1090,2150
13,Burger King,Double Stacker King,1050,1870


In [ ]:
# Combinar duas condições com o &

filtro = (df['Chain'] == 'Burger King') & (df['Calories']>1000)
display(df[filtro][['Item', 'Calories', 'Total Fat (g)', 'Sodium (mg)']])


,Item,Calories,Total Fat (g),Sodium (mg)
5,Triple Whopper Sandwich,1130,75.0,1120
6,Triple Whopper Sandwich with Cheese,1220,82.0,1470
8,Bacon King Sandwich,1150,79.0,2150
9,Cheddar Bacon King Sandwich,1190,84.0,1930
13,Double Stacker King,1050,68.0,1870
36,Spicy Chicken Nuggets - 20pc,1050,74.0,2840


In [ ]:
# Cocnhecendo as seções dos cardápios antes de classificá-los;
# E para isso, utilizamos a função de contagem de quantos vezes cada valor
# distinto aparece. value_counts()

print("Quantos itens por rede:")
print(df['Chain'].value_counts(), "\n")

print("Seções do Cardápio (Type)")
print(df['Type'].value_counts())

Quantos itens por rede:
Chain
Mc Donalds     325
Burger King    174
Name: count, dtype: int64 

Seções do Cardápio (Type)
Type
McCafe Coffees - Nonfat Milk    58
McCafe Coffees - Whole Milk     58
Beverages                       56
Breakfast                       55
Soft Drinks                     42
Desserts/Shakes                 40
Sandwiches                      34
Chicken & More                  18
Salads & Sides                  15
Flame Broiled Burgers           14
Chicken & Sauce                 13
Salads                          12
Additional Options              10
Desserts                         9
Whopper Sandwiches               8
Shakes/Smoothies                 6
Frappes                          6
Hot Coffees                      6
Iced Coffees                     6
McCafe Smoothies                 6
McCafe Frappes                   6
French Fries                     5
King Jr Meals - Entrees          5
Salad Dressings                  5
King Jr Meals - Beverages        

# Análise das Comparações: Quantidade de Itens de cada Rede x Seções p/Tipo no cardápio




1. O McDonalds tem quase o dobro de laches em relação ao Burguer King. Então, qualquer conta que junte os dois, o McDonalds vai sair com mais peso.

2. Se torna ainda mais grave, os cafés do McDonalds, está na base inteira, onde cada café contém duas versões de leite.

In [ ]:
# O que é mais viável fazer nessa situação é o seguinte:
# 1. Separar as bebidas das comidas e que vai se tornar primordial
# daqui p/frente;
# Como já rodamos value_counts() e conhecemos a lista COMPLETA de seções,
# podemos escrever a regra de forma explícita, com .isin(). Mais longo,
# e sem surpresa.
#
# Coluna criada:
#   df['Grupo'] -> 'Bebida' ou 'Comida'

SECOES_BEBIDA = [
    'McCafe Coffees - Nonfat Milk',
    'McCafe Coffees - Whole Milk',
    'Beverages',
    'Soft Drinks',
    'Hot Coffees',
    'Iced Coffees',
    'Frappes',
    'McCafe Frappes',
    'McCafe Smoothies',
    'Shakes/Smoothies',
    'King Jr Meals - Beverages',
]

# .isin() -> devolve True para os valores que está na lista. Que é o pertece á
bebidas = df['Type'].isin(SECOES_BEBIDA)

# np.where(condicao, valor1, valor2)
df['Grupo'] = np.where(bebidas, 'Bebida', 'Comida' )

# crosstab() -> cruza duas tabelas categóricas e conta as ocorrências.
print("Composição da Base:")
display(pd.crosstab(df['Chain'], df['Grupo'], margins=True, margins_name="Total"))

Composição da Base:


Grupo,Bebida,Comida,Total
Chain,,,
Burger King,69,105,174
Mc Donalds,184,141,325
Total,253,246,499


# Etapa 05 - Valores ausentes: onde suas posições diz mais que quantos são.

É o famoso `Nan`ao aparecer como resultado. Pois vem o questionamento: "*como irei preencher isso?*". Mais isso, é um passado errado, pois antres de tomar qualquer decisão, precisamos descobrir o **por que de sua ausência**, onde muita das vezes é o(s) bendito(s) símbolo(s) da formatação contábil do Excel e, que pode ter significados distintos.

**Esses significados distintos, pode ser:**

- **Aleatória:** Falha de registro sem padrão -> se imputar ou descartar é defensável.

- **Sistemática:** Onde a ausência tem uma causa e essa causa é a informação -> se preencher, destrói a informação.

In [ ]:
# Nessa célula, vamos contar os ausentes de cada coluna, com o uso de duas
# funções para verificação desses dados, que são:

# .isna() -> devolve True onde o valor está ausente (Nan);
# .sum() -> Conta os True

# Nan -> significa Not a number - sem número ou sem valor

ausente = df.isna().sum()

# Mostrar as colunas que tem os ausentes
print("Colunas com valores ausentes:")
print(ausente[ausente > 0], "\n")

# Fazer em proporção, pois costuma comunicar melhor que a brutagem bruta.
print("Em proporção com a base:")
print((ausente[ausente > 0] / len(df)).round(3))

Colunas com valores ausentes:
Serving Size (g)     258
Saturated Fat (g)     83
Trans Fat (g)         36
dtype: int64 

Em proporção com a base:
Serving Size (g)     0.517
Saturated Fat (g)    0.166
Trans Fat (g)        0.072
dtype: float64


# Análise das 3 colunas com base nos resultados.

Cada coluna, tem um contexto diferente. Vamos para a  análise.

`Satured Fat (g)`- com **82 ausentes** tem a equivalência de aproximadamente (17%), onde é considerado uma porcentagem baixa e tem sua análise concluída que é a presença dos traços contábil.
#
#
`Trans Fat (g)`- com **36 ausentes** tem a equivalência de aproximadamente (7%), onde é considerado uma porcentagem baixa e tem sua análise concluída que também é consequência da presença dos traços contábil.
#
#
`Serving Fat (g)`- com **258 ausentes** tem a equivalência de aproximadamente (52%). Agora, aqui precisamos ter cautela ao analisar, pois é uma porcentagem muita alta e representado por mais dad metade dos itens.


In [ ]:
# Com uma porcentagem muita alta no Serving Fat (g), precisamos analisar onde
# esses ausentes estão se concentrando.

# Para isso, precisamos seguir passos importante para chegar a conclusão:

# Passo 1: Traçar uma estratégia, onde cria uma coluna boolena marcando a
# ausência e depois agrupar.
# Assim, a média de uma coluna booleana é a PROPORÇÃO de True.

df['sem_porcao'] = df['Serving Size (g)'].isna()

print("Proporção de itens SEM peso de porção, por grupo:")
print(df.groupby('Grupo')['sem_porcao'].mean().round(3), "\n")

print("Proporção por rede:")
print(df.groupby('Chain')['sem_porcao'].mean().round(3))

Proporção de itens SEM peso de porção, por grupo:
Grupo
Bebida    1.00
Comida    0.02
Name: sem_porcao, dtype: float64 

Proporção por rede:
Chain
Burger King    0.397
Mc Donalds     0.582
Name: sem_porcao, dtype: float64


# Analisando os resultados alcançados

As bebidas têm 100% de ausência em Serving Size (g) porque sua unidade de medida (mL, onças) é diferente de gramas. Ao ver isso, a primeira suspeita seria erro de digitação ou falha de preenchimento — mas na verdade é regra de negócio: a coluna simplesmente não se aplica a bebidas. Isso também explica por que o McDonald's mostrava mais ausência que o Burger King: não é que ele "registra pior", é que tem mais bebida no cardápio. Diante disso, a decisão correta é não imputar e tratar isso como uma limitação documentada: não é possível comparar densidade calórica (kcal/g) entre bebidas e comidas, porque a base para esse cálculo (o peso em gramas) não existe para bebidas.

# Etapa 6 - Comparação justa utilizando `groupby`

Aqui é um dos principais comandos do Pandas, onde agregar dados e implementa um padrão de 3 passos para fazer uma comparação honesta. Esses 3 passos são:

- **Split** - quebra a a tabela em pedaços através de uma rede (pedaço);
- **Apply** - Aplica a função em cada pedaço;
- **Combine** - junta todos resultados em uma nova tabela.

**Vamos para à prática!**



In [ ]:
# Primeira Comparação: Separar as çinhas por rede e aplicar as funções

comparacao_1 = df.groupby('Chain')['Calories'].agg(['count', 'mean', 'median'])
print("SEM FILTRO - A base inteira")
display(comparacao_1.round(1))

SEM FILTRO - A base inteira


,count,mean,median
Chain,,,
Burger King,174,368.8,315.0
Mc Donalds,325,283.1,230.0


In [ ]:
# COMPARAÇÃO 2: filtrar uma única rede

# Primeiro filtra e depois agrupa, pois a ordem importa

# Filtrando:
so_comidas = df[df['Grupo'] == 'Comida']

# Agrupando:
comparacao_2 = so_comidas.groupby('Chain')['Calories'].agg(['count', 'mean', 'median', 'std'])
print("SÓ COMIDA - exceto Cafés e Refrigerantes:")
display(comparacao_2.round(1))


SÓ COMIDA - exceto Cafés e Refrigerantes:


,count,mean,median,std
Chain,,,,
Burger King,105,438.9,370.0,290.8
Mc Donalds,141,401.1,390.0,260.1


# O que aconteceu: Tirando bebida da conta, a diferença de calorias entre as redes caiu de 30% para 9%.

A pegadinha: Média e mediana discordam.

Média: Burger King é mais calórico (438,9 vs 401,1)
Mediana: McDonald's é mais calórico (390 vs 370)

Por quê: O desvio padrão do Burger King é maior (290,8 vs 260,1) — ou seja, ele tem alguns itens extremos (tipo combos gigantes) que puxam a média pra cima, mas não mexem na mediana. O McDonald's tem um cardápio mais uniforme.

Conclusão real: O item típico das duas redes tem calorias parecidas. A diferença não está no "normal" — está nos extremos do cardápio do Burger King.

A lição: Quando média e mediana discordam bastante, é sinal de distribuição assimétrica puxada por outliers — e é sempre a média que "engana" nesses casos, porque só ela é sensível a valores extremos.

In [ ]:
# Comparação 3: Comidas - somente sandíches, pois ainda em comidas, tem:
# Salada, sobremesas, batata e café da manhã.

# Os sandwichs
sanduiches = 'Sandwich|Burger|Whopper|Chicken'

# Filtragem
so_sanduiches = df[df['Type'].str.contains(sanduiches, case=False)]

# Agrupamento:
comparacao_3 = so_sanduiches.groupby('Chain')[['Calories', 'Sodium (mg)']].agg(['count', 'mean','median'])
print("SOMENTE SANDWICHES E FRANGO:")
display(comparacao_3.round(1))

SOMENTE SANDWICHES E FRANGO:


Calories               Sodium (mg)                
               count   mean median       count    mean  median
Chain                                                         
Burger King       40  620.2  630.0          40  1116.0  1080.0
Mc Donalds        47  384.5  390.0          47   877.9   830.0

# Etapa 07 - Distribuição e Variabilidade

Nessa etapa, será possível analisar que, toda discussão acima que trabalhamos acima cabe em um gráfico. Vamos ver a forma das distribuições e não apenas em resumos delas.

In [ ]:
# O principal objetivo desssa célula é: construir um histograma de calorias
# separado por grupos.

# O histograma responde: "como os valoress se distribuem?"
# Ele fatia o intervalo em caixas e conta quantos itens caem em cada uma.

# Para construir o histograma, utlizaremos a biblioteca: Plotly Express

import plotly.express as px

fig = px.histogram (
    df,
    x='Calories',
    color='Grupo',
    nbins=40,
    barmode='overlay',
    opacity=0.65,
    title='Distribuição de Calorias entre Bebidas x Comidas',
    labels={'Calories': 'Calorias (kcal)', 'count': 'Qunatidade de Itens'},
    color_discrete_map={'Bebida': '#4A7BA7', 'Comida': '#C8102E'},
)

fig.update_layout(template='plotly_white', height=420, bargap=0.05)
fig.show()

Aqui, quando analisamos nos deparamos com uma situação bem intessante. Pois as bebidas, está em um conjunto pequeno em e mais próximo de zero calorias e já as comidas se alarga para a direita com uma alta diversidade se tipos que contém uma variação muito grande de calorias.

Se juntar os dois grupos em uma média, chegrá em um resultado que, no McDonalds não terá nenhum produto real, pois sua média é descrita em 283 kcal. E essa prova está no gráfico, onde a comida com menos calorias está em cerca de 300 kcal.

In [ ]:
# Já aqui, teremos ccomo objetivo, ter a mesma comparação de somente comidas,
# mas no Gráfico Boxplot - em forma de figura.

# Específicações:
# linha do meio: mediana;
# caixa: q1 - 25%, q2 - 50% (média) e q3 - 75%
# hastes: até 1,5 x IQR
# pontos soltos: outliers

fig = px.box (
    so_comidas,
    x='Chain',                        # Redes: Mc x King
    y='Calories',
    color='Chain',
    points='outliers',
    title='Calorias por rede - somente comidas',
    labels={'Chain': '', 'Calories': 'Calorias (kcal)'},
    color_discrete_map={'Burger King': '#D62728', 'Mc Donalds': '#FFB000'},
)

fig.update_layout(template='plotly_white', height=420, showlegend=False)
fig.show()

# Medindo a variabilidade

O boxplot mostra a dispersão; agora vamos quantificá-la. Três medidas, e cada uma responde a uma pergunta diferente.

In [ ]:
# Nesta célula, vamos calculas as três medidas de dispersão por rede e entender
# para que cada uma serve.

def resumo_dispersao(serie):

  desvio = serie.std()
  q1 = serie.quantile(0.25)
  q3 = serie.quantile(0.75)
  iqr = q3 - q1
  cv = desvio / serie.mean()

  return {'desvio_padrao': desvio, 'iqr': iqr, 'cv': cv}

# Aplicando a função em cada rede, dentro da comida:
for rede in ['Burger King', 'Mc Donalds']:
    dados_redes = so_comidas.loc[so_comidas['Chain'] == rede, 'Calories']
    r = resumo_dispersao(dados_redes)
    print(f"{rede:14s} | DP = {r['desvio_padrao']:6.1f} | "
          f"IQR {r['iqr']:6.1f} | CV = {r['cv']:.2f}")

Burger King    | DP =  290.8 | IQR  400.0 | CV = 0.66
Mc Donalds     | DP =  260.1 | IQR  350.0 | CV = 0.65


# Etapa 08 - Correlação Linear

Correlação Linear, responde uma pergunta específica e limitada. Pois **quando uma variável sobe, a outra tende a subir junta?**

Para responder essa pergunta, para calcular o coeficiente utilizamos a de **Pearson** que tem seu valor ente +1 e -1 e esses números tem a seguinte representação:

- +1: coeficiente linear e perfeito;
- 0: nenhuma relação linear;
- -1: coeficiente linear perfeita e descrescente.

# Fórmula

$$ r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2} \cdot \sqrt{\sum (y_i - \bar{y})^2}} $$

**Numerador:** Para cada item, deve calcular o quanto ele está acima ou abaixo da média em X e o quanto está acima ou abaixo da média em Y e multiplica os dois desvios.

 - Acima da média em ambas -> + / + = contribuição +;
 - Abaixo da média em ambas -> - / - = contribuição +;
 - Item Acima e outra Abaixo -> contribuição -.

 Somando tudo: se os desvios andam juntos, a soma cresce. O denominador só normaliza o resultado para a faixa de −1 a +1, e é ele que faz o coeficiente não depender da unidade de medida.

In [ ]:
# Vamos calcular a correlação de Pearson na mão, com Numpy, seguindo termo a
# termo da fórmula.

def correlacao_pearson(x, y):

  # Iremos utilzar a função .np.asarray -> aceita lista, serie e array, mas de
  # devolve somente array.
  x = np.asarray(x, dtype=float)
  y = np.asarray(y, dtype=float)

  # Desvios em relação a média. Broadcasting em ação, onde  X e Y são um escalar
  # e é subtraído por cada posição do vetor.
  dx = x - x.mean()
  dy = y - y.mean()

  # Soma dos produtos do NUMERADOR:
  numerador = np.sum(dx * dy)

  # Produto das raízes da soma dos quadrados, DENOMINADOR:
  denominador = np.sqrt(np.sum(dx ** 2)) * np.sqrt(np.sum(dy ** 2))

  return numerador / denominador

# ------ Conferência: da mão x Pandas ---------- #
x = df['Total Fat (g)']
y = df['Calories']

na_mao = correlacao_pearson(x, y)
biblioteca_pandas = x.corr(y)

print(f"Na mão           : {na_mao:.10f}")
print(f"Biblioteca Pandas: {biblioteca_pandas:.10f}")
print(f"Diferença:       : {abs(na_mao - biblioteca_pandas):.2e}")


Na mão           : 0.8500762627
Biblioteca Pandas: 0.8500762627
Diferença:       : 7.77e-16


# A matriz de correlação

Com uma função para dois vetores, o passo natural é cruzar todos com todos. O .corr() do pandas faz isso de uma vez.

In [ ]:
# Matrix curricular de todas as colunas númericas, apresentando em mapa de calor.

# Leitura:
# -> diagonal sempre será 1, pois toda variável é correlacionada a si mesma;
# -> É simétrica, o triangulo encima é o mesmo embaixo;
# -> vermelho forte, correlação positiva alta - Itens Acima.

# Selcionado todas as tabelas númericas com slects_dtypes(include="number"),
# mas traria com Calories from fat, que nesse caso é uma função direta do Total Fat
# onde o par teria 1,00 e pouparia espaços desnecessários na tela e não informaria
# nada.

COLUNAS = [
    'Calories', 'Serving Size (g)', 'Total Fat (g)', 'Saturated Fat (g)',
    'Total Carb (g)', 'Total Sugar (g)', 'Protein (g)',
    'Sodium (mg)', 'Chol (mg)', 'Dietary Fiber (g)',
]

# Descartar pares Incompletos: .corr(), percorre coluna a coluna e descarta
# automaticamente.

matriz = df[COLUNAS].corr()

fig = px.imshow (
    matriz.round(2),
    text_auto=True,
    aspect='auto',
    color_continuous_scale='dense',
    zmin=-1, zmax=1,
    title='Matriz Correlação -  Todas as Colunas Numéricas',
)

fig.update_layout(template='plotly_white', height=620)
fig.show()

In [ ]:
# Entendendo porque a açucar tem correlação apenas de 0,33, enquanto o sódio
# Tem uma correlação de 0.72 que não tem caloria alguma?!

# Investigação com cálculo separado por cada grupo.
# Movimento Análitico - número global não faz sentido, DESAGREGUE.

print("Correlação entre Açúcar x Calorias\n")

# Na base inteira, tudo misturado:
geral = df['Total Sugar (g)'].corr(df['Calories'])
print(f"Base Inteira (n={len(df)})           : r = {geral:.2f}")

# Dentro de cada elemento:
for grupo in ['Bebida', 'Comida']:
  sub = df[df['Grupo'] == grupo]
  r = sub['Total Sugar (g)'].corr(sub['Calories'])
  rotulo = f"Só {grupo} (n={len(sub)})"
  print(f" {rotulo:31s}: r = {r:.2f}")

Correlação entre Açúcar x Calorias

Base Inteira (n=499)           : r = 0.33
 Só Bebida (n=253)              : r = 0.87
 Só Comida (n=246)              : r = 0.41


Leia de novo, devagar, porque o resultado é contraintuitivo: a correlação da base inteira é menor que a de cada um dos dois grupos separadamente. Não é uma média dos dois. É menor que os dois.

Isso parece impossível e não é. Quando você junta duas nuvens de pontos que ocupam regiões diferentes do gráfico, a relação entre as nuvens compete com a relação dentro de cada uma — e pode cancelá-la. As bebidas têm muito açúcar e poucas calorias; as comidas têm pouco açúcar e muitas calorias. Entre os dois grupos, a relação é negativa, e é ela que come a correlação positiva que existe dentro de cada um.

E os números separados contam uma história física perfeitamente coerente:

Nas bebidas, r = 0,87. Num refrigerante ou num frappé, o açúcar praticamente é a caloria — não há gordura nem proteína competindo. Mais açúcar, mais caloria, quase linearmente.
Nas comidas, r = 0,41. Num Whopper, a caloria vem da gordura e da proteína. O açúcar do pão e do molho contribui pouco.
O 0,33 da base inteira não descreve nenhum dos dois. É um artefato da mistura.

In [ ]:
# Ver esse mesmo achado só que em gráfico:

fig = px.scatter(
    df,
    x='Total Sugar (g)',
    y='Calories',
    color='Grupo',
    trendline='ols',
    opacity=0.7,
    title='Açúcar x Calorias: duas nuvens, duas histórias',
    labels={'Total Sugar (g)': 'Açúcar (g)', 'Calories': 'Calorias (kcal)'},
    color_discrete_map={'Bebida': '#4A7BA7', 'Comida': '#C8102E'},
)

fig.update_layout(template='plotly_white', height=460)
fig.show()

---
# Etapa 9 — Desafio (entrega obrigatória)

Três TODOs e uma análise escrita. Cada TODO tem um teste automático logo abaixo: rode a célula e ele avisa se a resposta está certa.

## 🛠️ TODO 1 — Correlação de Pearson na mão, versão robusta

A função `correlacao_pearson` da Etapa 8 tem um defeito: se algum dos vetores tiver valores ausentes (`NaN`), a resposta sai `nan`. Isso não é hipotético — a coluna `Serving Size (g)` tem 258 ausentes, e ninguém consegue correlacioná-la com nada usando a nossa versão.

Escreva `correlacao_limpa(x, y)`, que **descarta os pares em que qualquer um dos dois valores está ausente** e só então aplica a fórmula.

**Lógica necessária:**
1. Converta `x` e `y` para arrays de `float`.
2. Construa uma máscara com `~(np.isnan(x) | np.isnan(y))` — `True` onde **nenhum** dos dois é `NaN`.
3. Aplique a máscara aos dois vetores.
4. Devolva `correlacao_pearson()` dos vetores filtrados.

> **Atenção ao passo 2:** o descarte tem que ser **em par**. Se você limpar `x` e `y` separadamente, eles ficam com tamanhos diferentes e desalinhados — e o resultado sai errado sem dar erro.

In [ ]:
# =====================================================================
# TODO 1 — Complete a função abaixo.
#
# correlacao_limpa(x, y) deve descartar os PARES com valor ausente e
# devolver a correlação de Pearson do que sobrou.
#
# Dica: o operador ~ inverte um vetor de booleanos (True vira False).
#       O operador | é o "ou" elemento a elemento.
# =====================================================================

def correlacao_limpa(x, y):
    """Correlação de Pearson descartando pares com valores ausentes.

    Parâmetros:
        x (array-like): primeiro vetor numérico, pode conter NaN.
        y (array-like): segundo vetor numérico, do mesmo tamanho de x.

    Devolve:
        float: o coeficiente r calculado só sobre os pares completos.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = ~(np.isnan(x) | np.isnan(y))
    x_limpo = x[mask]
    y_limpo = y[mask]

    return correlacao_pearson(x_limpo, y_limpo)


# ------------------- teste automático, não precisa alterar -------------------
_r = correlacao_limpa(df['Serving Size (g)'], df['Calories'])

if _r is None:
    print("❌ TODO 1 ainda não foi resolvido.")
elif np.isnan(_r):
    print("❌ Saiu NaN: a máscara não está descartando os ausentes.")
elif np.isclose(_r, 0.8503, atol=1e-3):
    # Segundo teste: um caso pequeno, montado na mão, para garantir que
    # a função não está devolvendo um valor fixo.
    _teste = correlacao_limpa([1, 2, np.nan, 4], [2, 4, 99, 8])
    if np.isclose(_teste, 1.0):
        print(f"✅ TODO 1 correto! r = {_r:.4f} e o caso de teste passou.")
    else:
        print(f"⚠️ Quase: o caso pequeno deveria dar 1.0 e deu {_teste}.")
else:
    print(f"❌ Resultado inesperado: {_r:.4f} — esperado ≈ 0.8503.")

✅ TODO 1 correto! r = 0.8503 e o caso de teste passou.


## 🛠️ TODO 2 — A comparação justa, em função

Refaça a Comparação 3 da Etapa 6, mas como uma função reutilizável, capaz de comparar as duas redes em **qualquer** seção do cardápio.

Escreva `comparar_redes(dados, padrao, coluna)`, que:

1. Filtra `dados` mantendo as linhas cujo `Type` contenha `padrao` (use `case=False`);
2. Agrupa o resultado por `Chain`;
3. Devolve um `DataFrame` com `count`, `mean` e `median` da `coluna` pedida.

In [ ]:
# =====================================================================
# TODO 2 — Complete a função abaixo.
#
# Deve devolver um DataFrame com uma linha por rede e as colunas
# count, mean e median.
#
# Dica: dados[mascara].groupby('Chain')[coluna].agg([...])
# =====================================================================

def comparar_redes(dados, padrao, coluna='Calories'):
    """Compara as duas redes dentro de uma seção do cardápio.

    Parâmetros:
        dados (pd.DataFrame): a base completa.
        padrao (str): trecho de texto a procurar na coluna Type.
        coluna (str): a coluna numérica a resumir. Padrão: 'Calories'.

    Devolve:
        pd.DataFrame: uma linha por rede, com count, mean e median.
    """
    mascara_2 = dados['Type'].str.contains(padrao, case=False, na=False)
    resultado = dados[mascara_2].groupby('Chain')[coluna].agg(['count', 'mean', 'median'])
    return resultado



# ------------------- teste automático, não precisa alterar -------------------
_res = comparar_redes(df, 'Sandwich|Burger|Whopper|Chicken')

if _res is None:
    print("❌ TODO 2 ainda não foi resolvido.")
elif not isinstance(_res, pd.DataFrame):
    print(f"❌ A função devolveu {type(_res).__name__}, esperado DataFrame.")
elif list(_res.columns) != ['count', 'mean', 'median']:
    print(f"❌ Colunas erradas: {list(_res.columns)}")
elif len(_res) != 2:
    print(f"❌ Esperado 2 linhas (uma por rede) e vieram {len(_res)}.")
elif np.isclose(_res.loc['Burger King', 'mean'], 620.2, atol=0.5):
    # Segundo teste: outra seção, para garantir que o filtro funciona.
    _sobremesa = comparar_redes(df, 'Dessert')
    if len(_sobremesa) == 2:
        print(f"✅ TODO 2 correto! BK = "
              f"{_res.loc['Burger King', 'mean']:.1f} kcal, Mc = "
              f"{_res.loc['Mc Donalds', 'mean']:.1f} kcal.")
    else:
        print("⚠️ Quase: o filtro falhou quando testado com 'Dessert'.")
else:
    print(f"❌ Média do BK saiu {_res.loc['Burger King', 'mean']:.1f}, "
          f"esperado ≈ 620.2.")

✅ TODO 2 correto! BK = 620.2 kcal, Mc = 384.5 kcal.


## 🛠️ TODO 3 — Auditoria pela regra física

Retomando a Aula 2: gordura fornece **9 kcal por grama**. A base tem duas colunas que precisam obedecer a essa constante — `Calories from fat` e `Total Fat (g)`. Dividindo a primeira por 9, o resultado tem que bater com a segunda.

Na Aula 2 você aplicou essa regra na mão, em 25 linhas, e encontrou dois registros impossíveis. Agora são 499 linhas e a conta tem que ser vetorizada.

Escreva `auditar_gordura(dados, tolerancia)`, que:

1. Calcula `Calories from fat / 9`;
2. Calcula o **resíduo**: a diferença absoluta entre esse valor e `Total Fat (g)`;
3. Devolve as linhas cujo resíduo passa da `tolerancia`, com as colunas `Chain`, `Item`, `Calories from fat`, `Total Fat (g)` e uma coluna nova chamada `residuo`.

> Não presuma o resultado. A resposta pode muito bem ser "nenhuma linha" — e, se for, isso também é um achado que você precisa saber interpretar.

In [ ]:
# =====================================================================
# TODO 3 — Complete a função abaixo.
#
# Dica: para criar a coluna residuo sem alterar o DataFrame original,
#       use .copy() antes de atribuir. Alterar uma fatia de um
#       DataFrame gera o aviso SettingWithCopyWarning.
# =====================================================================

def auditar_gordura(dados, tolerancia=1.0):
    """Procura itens em que as calorias de gordura não batem com a gordura.

    Regra física: 1 g de gordura fornece 9 kcal, logo
    (Calories from fat / 9) deve ser igual a Total Fat (g).

    Parâmetros:
        dados (pd.DataFrame): a base completa.
        tolerancia (float): diferença máxima aceitável, em gramas.

    Devolve:
        pd.DataFrame: as linhas suspeitas, com a coluna 'residuo'.
    """
    tabela = dados.copy()

    tabela['gordura_implicita'] = tabela['Calories from fat'] / 9
    tabela['residuo'] = (tabela['gordura_implicita'] - tabela['Total Fat (g)']).abs()

    suspeitos = tabela[tabela['residuo'] > tolerancia]

    return suspeitos[['Chain', 'Item', 'Calories from fat', 'Total Fat (g)', 'residuo']]

    # ------------------------------------------------------------------
    # TODO 3: calcule a gordura implícita, o resíduo absoluto, e devolva
    #         as linhas em que o resíduo passa da tolerância, com as
    #         colunas: Chain, Item, Calories from fat, Total Fat (g),
    #         residuo — nessa ordem.
    # ------------------------------------------------------------------




# ------------------- teste automático, não precisa alterar -------------------
_sus = auditar_gordura(df, tolerancia=1.0)

if _sus is None:
    print("❌ TODO 3 ainda não foi resolvido.")
elif not isinstance(_sus, pd.DataFrame):
    print(f"❌ A função devolveu {type(_sus).__name__}, esperado DataFrame.")
elif 'residuo' not in _sus.columns:
    print("❌ Falta a coluna 'residuo' no resultado.")
elif len(_sus) == 1 and np.isclose(_sus['residuo'].iloc[0], 2.222, atol=1e-2):
    # Segundo teste: com tolerância menor, mais linhas devem aparecer.
    _apertado = auditar_gordura(df, tolerancia=0.5)
    if len(_apertado) == 76:
        print(f"✅ TODO 3 correto! Com tolerância 1,0 g apareceu "
              f"{len(_sus)} item; com 0,5 g apareceram {len(_apertado)}.")
        display(_sus)
    else:
        print(f"⚠️ Quase: com tolerância 0,5 esperava 76 linhas, "
              f"vieram {len(_apertado)}.")
else:
    print(f"❌ Com tolerância 1,0 esperava 1 linha e vieram {len(_sus)}.")

✅ TODO 3 correto! Com tolerância 1,0 g apareceu 1 item; com 0,5 g apareceram 76.


,Chain,Item,Calories from fat,Total Fat (g),residuo
80,Burger King,Double Croissan'wich with Sausage & Bacon,340,40.0,2.222222


### Sua análise — edite esta célula

Depois de resolver o TODO 3, você tem um resultado para interpretar. Escreva de **6 a 10 linhas** respondendo, nesta ordem:

1. **O que a auditoria encontrou.** Quantos itens passaram da tolerância de 1,0 g? E de 0,5 g? Por que o número muda tanto entre as duas tolerâncias?

2. **A conclusão sobre a qualidade da base.** Na Aula 2, a mesma regra aplicada a 25 linhas encontrou dois registros fisicamente impossíveis. Aqui, em 499 linhas, o resultado foi outro. O que isso diz sobre esta base? E o que a *ausência* de erro te autoriza a concluir — e o que **não** te autoriza?

3. **A decisão de analista.** Suponha que você precise entregar a comparação entre as redes ao Burger King, que quer decidir onde investir. Qual das três comparações da Etapa 6 você entregaria, e que ressalva escreveria junto? Cite ao menos um número da Etapa 5 ou 6 para sustentar sua escolha.

---

1. Análise do que a Auditoria encontrou: **Um único item esta tem tolerância de 1,0, todos os demais é de 0,5. E isso se dar, pelo fato de ter um valor muito alto de residuos, onde é impossível de ser arredondado. Pois os rótulos nutricionais. E com isso, aproximado de 1 é possível arredondar, mas acima disso é impossível.**

2. Conclusão sobre Qualidade  de Base: **A auditoria automatizada realizada no TODO 3 não encontrou nenhuma inconsistência nos 499 registros analisados (resíduo $\le 0{,}5\text{ g}$). Diferente da amostra reduzida da Aula 2, a relação entre calorias e gordura respeita a regra física de $9\text{ kcal/g}$ em toda a extensão do conjunto de dados. Com isso, confirma que, não obteve erros grossoeiros de digitação em que afetasse diretamente na realização dos cálculos. POrtanto, não autoriza que esses dados calculados não representa fisicamente na prática, pois são informações de dados para afins de rotulagem. E , vale resaltar que, podem contêr problemas desde a base de dados.**

3. A descisão do Analista: **Decisão: Eu entregaria a Comparação 2 (Só Comidas).
Justificativa: A primeira comparação é enviesada, pois a base de uma das redes inclui bebidas (refrigerantes) e a outra não, distorcendo a média geral. Filtrando apenas as comidas, temos uma visão mais justa do cardápio sólido.
Os dados mostram que a média calórica das comidas do Burger King é de 438.9 kcal, enquanto a do McDonald's é de 401.1 kcal. A ressalva é que essa análise ainda mistura acompanhamentos e sobremesas com o prato principal. Como recomendação de investimento, o Burger King poderia focar no desenvolvimento de hambúrgueres menos calóricos e mais saudáveis, aproximando-se dos valores do concorrente para atingir um público mais amplo, além de estruturar melhor seus combos de bebidas.**
---

**Nota de Autoria:** Todo o raciocínio analítico, escolha de métricas e interpretações de negócios apresentados neste relatório são de minha própria autoria. O uso de Inteligência Artificial foi aplicado exclusivamente como ferramenta de revisão ortográfica, estruturação e organização textual das ideias.
